# Portique : entraînement YOLO Pose sur GPU
Ce notebook charge les poids sauvegardés du PC et lance 100 nouveaux passages avec un optimiseur neuf.
Transférer portique_dataset.zip dans Mon Drive/PortiqueVision, choisir un GPU et exécuter les cellules.
Les résultats sont sauvegardés dans Drive/runs/container-pose-gpu (ou un dossier numéroté).
Après interruption, ne pas relancer aveuglément cette cellule : utiliser le dernier checkpoint GPU pour une reprise.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.174'])
import torch
assert torch.cuda.is_available(), 'Active le GPU dans Exécution → Modifier le type d’exécution.'
print('GPU :', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/PortiqueVision')
ARCHIVE = DRIVE / 'portique_dataset.zip'
assert ARCHIVE.exists(), 'Dépose portique_dataset.zip dans Mon Drive/PortiqueVision.'
DATA = Path('/content/portique_dataset')
RUN = DRIVE / 'runs/container-pose'
RUN.mkdir(parents=True, exist_ok=True)


In [ ]:
import zipfile, math, json, yaml, cv2
DATA.mkdir(exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as pack:
    for name in pack.namelist():
        (DATA / name).resolve().relative_to(DATA.resolve())
    pack.extractall(DATA)
for split, expected in [('train',314), ('val',105)]:
    images = sorted((DATA / 'images' / split).glob('*.png'))
    assert len(images) == expected
    for image in images:
        assert cv2.imread(str(image)) is not None, str(image)
        for line in (DATA / 'labels' / split / (image.stem + '.txt')).read_text().splitlines():
            values = list(map(float, line.split()))
            assert len(values) == 17 and values[0] == 0 and all(math.isfinite(v) for v in values)
            assert all(0 <= v <= 1 for v in values[1:5])
            for j in range(5,17,3):
                assert 0 <= values[j] <= 1 and 0 <= values[j+1] <= 1 and values[j+2] in (0,2)
    print(split, len(images), 'images vérifiées')
CONFIG = DATA / 'container-pose.yaml'
CONFIG.write_text(yaml.safe_dump(dict(path=str(DATA),train='images/train',val='images/val',names={0:'container'},kpt_shape=[4,3])))


In [ ]:
from ultralytics import YOLO
import torch

assert torch.cuda.is_available(), "Active le GPU dans Colab."
checkpoint = DRIVE / "runs/container-pose/weights/last.pt"
if not checkpoint.exists():
    checkpoint = DATA / "checkpoint/last.pt"

# Conserver les poids appris ; reconstruire un optimiseur neuf sur Colab.
model = YOLO(str(checkpoint))
model.reset_callbacks()
results = model.train(
    resume=False,
    data=str(CONFIG),
    epochs=100,
    device=0,
    batch=4,
    workers=2,
    amp=False,
    optimizer="AdamW",
    cache=False,
    plots=False,
    save_period=-1,
    project=str(DRIVE / "runs"),
    name="container-pose-gpu",
)
RUN = Path(model.trainer.save_dir)
print("Résultats sauvegardés dans :", RUN)


In [ ]:
BEST = RUN / 'weights/best.pt'
assert BEST.exists(), 'Le meilleur modèle n’est pas encore sauvegardé.'
evaluation = YOLO(str(BEST)).val(data=str(CONFIG), device=0, batch=4,
                               plots=False, project=str(DRIVE / 'evaluation'), name='synthetic')
summary = {key: float(value) for key, value in evaluation.results_dict.items()}
(DRIVE / 'validation-synthetic.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
print('Modèle à récupérer :', BEST)


In [ ]:
from google.colab import files
files.download(str(BEST))
